Inferential analysis script
---------------------------
This script performs the regressions used for the thesis. Independent variables and optional expanded features can be selected in the configuration section.

In [40]:
import os
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import numpy as np
import openpyxl

configuration:

In [ ]:
INPUT_DIR = "."
INPUT_FILE = "Final Dataset.csv"
OUTPUT_FILE = "Regressions results.xlsx"

# Target variable
TARGET = 'CONTR'

# Independent variables
FEATURES = {
    # The readability scores per subcriteria (indexes 0 through 11)
    'readability subcrit scores': [s + "_score" for s in ['A_01', 'A_02', 'B_01', 'B_02', 'B_03', 'C_01', 'C_02', 'C_04', 'C_05', 'C_06', 'D_01', 'L_01']],
    # The readability scores per main criteria (indexes 0 through 4)
    'readability main crit scores': ['Crit_' + s for s in['A', 'B', 'C', 'D', 'L']],
    # The average readability score (LLM) and average Gunning Fog score (indexes 0 through 1)
    'readability average scores': ['RDBL', 'Gunning_Fog'],
    # The GAP score
    'GAP': ['GAP']
    }['GAP'][0]    # Choose which feature(s) to test here

# Control variables
CONTROLS = ['CAPln', 'LEV', 'BOARD', 'WORDSln']

# Advanced features (are standardized in the regression script) (can be empty)
INTERACTION_FEATURES = []
QUADRATIC_FEATURES = []

main algorithm:

In [ ]:
def load_and_prepare_data(all_needed_variables):
    # Construct the full file path
    file_path = os.path.join(INPUT_DIR, INPUT_FILE)
    print(f"Loading data from: {file_path}")

    # Load the dataset
    df = pd.read_csv(file_path, sep=",")

    # Clean the data: Convert selected columns to numeric, forcing errors to NaN
    print("Converting columns to numeric values and cleaning data...")
    for col in all_needed_variables:
        if col in df.columns:
            # errors='coerce' turns non-numeric values (like text or spaces) into NaN
            df[col] = pd.to_numeric(df[col], errors='coerce')
        else:
            raise KeyError(f"Column '{col}' not found in the dataset.")

    # Drop rows that contain any missing values (NaN) in our selected variables
    initial_row_count = len(df)
    df = df.dropna(subset=all_needed_variables)
    cleaned_row_count = len(df)
    
    print(f"Dropped {initial_row_count - cleaned_row_count} rows containing missing or non-numeric values.")

    # Re-isolate features and target after cleaning
    independent_vars = [var for var in all_needed_variables if var != TARGET]    
    X = df[independent_vars].copy()
    y = df[TARGET]

    # Add quadratic features if specified
    if QUADRATIC_FEATURES: 
        for var in QUADRATIC_FEATURES:
            # Standardize first using z-scores
            mean_val = df[var].mean()
            std_val = df[var].std()
            df[var] = (df[var] - mean_val) / std_val

            col_name = f"{var}_squared"
            X[col_name] = df[var] ** 2
            print(f"Added quadratic feature: {col_name}")

    # Add interaction features if specified
    if INTERACTION_FEATURES:
        for var1, var2 in INTERACTION_FEATURES:
            # Standardize first using z-scores
            for var in (var1, var2):
                mean_val = df[var].mean()
                std_val = df[var].std()
                df[var] = (df[var] - mean_val) / std_val

            col_name = f"{var1}_x_{var2}"
            X[col_name] = df[var1] * df[var2]
            print(f"Added interaction feature: {col_name}")

    # Add a constant term (intercept) to the predictor variables
    X = sm.add_constant(X)

    # Print descriptives
    print(df[all_needed_variables].describe().transpose()[['count', 'mean', 'std', 'min', 'max']])

    return X, y, df

def main():
    try:
        # Combine all basic independent variables and the target
        features = FEATURES if isinstance(FEATURES, list) else [FEATURES]   # Convert feature to list if only a single feature is used
        all_needed_variables = features + CONTROLS + [TARGET]
        # Prepare the features and target
        X, y, df = load_and_prepare_data(all_needed_variables)

        # Fit the logistic (Logit) regression model
        print("\nTraining the regression model...")
        model = sm.Logit(y, X).fit()
        

        # Print the comprehensive regression results
        print("\n" + "=" * 80)
        print("REGRESSION MODEL RESULTS")
        print("=" * 80)
        print(model.summary())

        # Print Odds Ratios and Pearson residuals
        odds_ratios = np.exp(model.params)
        print(odds_ratios)
        # Residuals larger (smaller) than 3 are viewed as outliers
        outliers = np.where(np.abs(model.resid_pearson) > 3)[0]
        print(f"Potential outliers index: {outliers}")

        # Calculate VIF values
        print("\n" + "=" * 40)
        print("MULTICOLLINEARITY CHECK (VIF)")
        print("=" * 40)
        
        vif_data = []
        vif_dict = {}
        for i, col_name in enumerate(model.model.exog_names):
            if col_name != 'const':  # Skip intercept
                try:
                    vif_value = variance_inflation_factor(model.model.exog, i)
                    vif_value_rounded = round(vif_value, 3)
                    vif_data.append({"Variable": col_name, "VIF": vif_value_rounded})
                    vif_dict[col_name] = vif_value_rounded
                except Exception as e:
                    vif_data.append({"Variable": col_name, "VIF": "Error/Constant"})
                    vif_dict[col_name] = "Error"
            else:
                vif_dict['const'] = np.nan  

        # Print VIF
        vif_df = pd.DataFrame(vif_data)
        if not vif_df.empty:
            print(vif_df.to_string(index=False))
        print("=" * 40)
    
        # --- EXPORT to Excel--- (Dutch)
        # Descriptives
        desc_stats = df[all_needed_variables].describe().transpose()[['count', 'mean', 'std', 'min', 'max']]

        # Individual results
        results_df = pd.DataFrame({
            'Coefficient (Log-Odds)': model.params,
            'Standard Error': model.bse,
            'z-value': model.tvalues,
            'p-value': model.pvalues,
            'Odds Ratio (OR)': np.exp(model.params)
        })
        results_df['VIF'] = results_df.index.map(vif_dict)

        # Model results
        pseudo_r2 = model.prsquared
        llr_p_value = model.llr_pvalue
        outlier_indices = df.index[np.abs(model.resid_pearson) > 3].tolist()
        
        model_metrics = pd.DataFrame({
            'Modelmetriek': [
                'Pseudo R-squared (McFadden)', 
                'LLR p-value', 
                'Aantal potentiële outliers',
                'Indexen van outliers'
            ],
            'Waarde': [
                round(pseudo_r2, 4),
                f"{llr_p_value:.4e}" if llr_p_value < 0.001 else round(llr_p_value, 4),
                len(outlier_indices),
                str(outlier_indices)
            ]
        })

        # Compile to one dataframe
        sheet = f"{str(FEATURES)}-{str(INTERACTION_FEATURES)}-{str(QUADRATIC_FEATURES)}".replace("[","").replace("]","")[:31]

        witregel = pd.DataFrame()

        individual_metrics = pd.concat([results_df, desc_stats], axis=1)
        export_list = [
            individual_metrics,
            witregel, witregel,
            model_metrics
        ]
        
        final_dataframe = pd.concat(export_list, axis=0)

        with pd.ExcelWriter(OUTPUT_FILE, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
            final_dataframe.to_excel(writer, sheet_name=sheet, index=True)
            
    
    except FileNotFoundError:
        print(
            f"Error: The file '{INPUT_FILE}' was not found in the directory '{INPUT_DIR}'."
        )
    except KeyError as e:
        print(
            f"Error: One of the specified variables does not exist in the dataset: {e}"
        )
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

if __name__ == "__main__":
    main()